# Módulo 3: Agentes de IA
- **Sesión:** 02 — Frameworks de agentes: LangGraph y protocolo MCP
- **Laboratorio básico:** Un agente con LangGraph
- **Fecha:** 28/10/2026

In [ ]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
import os
import logging

logging.getLogger("google_genai.models").setLevel(logging.ERROR)

from langchain.tools import tool
from typing import TypedDict
from langgraph.graph import StateGraph, MessagesState, START, END



In [ ]:
# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

# Configuración del modelo de lenguaje
load_dotenv(override = True)

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)


In [ ]:
def ver_grafo(app):
    """Muestra el diagrama del grafo como imagen; si no hay conexión, lo imprime en formato Mermaid."""
    try:
        from IPython.display import Image, display
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

def mostrar(mensajes):
    """Imprime el tipo y el contenido de cada mensaje."""
    for m in mensajes:
        if getattr(m, "tool_calls", None):
            texto = "solicita " + ", ".join(f"{t['name']}({t['args']})" for t in m.tool_calls)
        else:
            texto = m.text
        print(f"  [{type(m).__name__:12}] {texto[:120]}")

## Datos


In [ ]:

PEDIDOS = {
    "48213": "en ruta, llega el 26/10/2026 a Arequipa",
    "77100": "entregado el 10/10/2026 en Lima",
    "50555": "en preparación, llega el 30/10/2026 a Cusco",
}

@tool
def consultar_pedido(numero_pedido: str) -> str:
    """Consulta el estado de un pedido de TiendaYa a partir de su número."""
    return PEDIDOS.get(numero_pedido, f"No existe el pedido {numero_pedido}.")

print(consultar_pedido.invoke({"numero_pedido": "48213"}))


---
## Ejercicio 1 — Mensajes que se acumulan

### Concepto
En el notebook anterior, cuando un nodo devolvía una clave, el valor nuevo **reemplazaba** al anterior. En una conversación los mensajes deben **acumularse**.

`MessagesState` es un estado ya definido con una clave `messages` que **agrega** los mensajes nuevos al final de la lista (reductor `add_messages`). Este ejercicio no usa el modelo: dos nodos agregan un mensaje cada uno.

### Ejecutar

In [ ]:
def nodo_a(estado: MessagesState) -> dict:
    return {"messages": [("ai", "Mensaje escrito por el nodo A")]}


def nodo_b(estado: MessagesState) -> dict:
    return {"messages": [("ai", "Mensaje escrito por el nodo B")]}


grafo = StateGraph(MessagesState)
grafo.add_node("a", nodo_a)
grafo.add_node("b", nodo_b)
grafo.add_edge(START, "a")
grafo.add_edge("a", "b")
grafo.add_edge("b", END)

r = grafo.compile().invoke({"messages": [("user", "Mensaje del cliente")]})
mostrar(r["messages"])

In [ ]:
app = grafo.compile()
ver_grafo(app)

---
## Ejercicio 2 — Un chat como grafo de un nodo

### Concepto
Un chat es un grafo con un solo nodo, `modelo`, que recibe el historial y **agrega** la respuesta del modelo.

```
START ──▶ modelo ──▶ END
```

Sin herramientas, el modelo no puede consultar el sistema de pedidos.

### Ejecutar

In [ ]:
def modelo_simple(estado: MessagesState) -> dict:
    return {"messages": [llm.invoke(estado["messages"])]}


grafo = StateGraph(MessagesState)
grafo.add_node("modelo", modelo_simple)
grafo.add_edge(START, "modelo")
grafo.add_edge("modelo", END)
chat = grafo.compile()

r = chat.invoke({"messages": [("user", "¿Dónde está mi pedido 48213?")]})
mostrar(r["messages"])

In [ ]:
app = grafo.compile()
ver_grafo(app)

---
## Ejercicio 3 — Herramientas: el ciclo del agente

### Concepto
Para que el modelo use herramientas se necesitan tres piezas:

| Pieza | Función |
|---|---|
| `llm.bind_tools([...])` | El modelo conoce las herramientas y puede solicitarlas |
| `ToolNode([...])` | Nodo que **ejecuta** las herramientas solicitadas |
| `tools_condition` | Arista condicional: si el modelo solicitó una herramienta, va a `tools`; si no, termina |

La arista `tools → modelo` cierra el **ciclo**: el modelo recibe el resultado y decide si necesita algo más o si ya puede responder.

```
START ──▶ modelo ──(solicita herramienta)──▶ tools
            ▲  │                               │
            │  └──(responde)──▶ END            │
            └──────────────────────────────────┘
```

### Ejecutar

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

HERRAMIENTAS = [consultar_pedido]
llm_con_herramientas = llm.bind_tools(HERRAMIENTAS)


def modelo(estado: MessagesState) -> dict:
    return {"messages": [llm_con_herramientas.invoke(estado["messages"])]}


grafo = StateGraph(MessagesState)
grafo.add_node("modelo", modelo)
grafo.add_node("tools", ToolNode(HERRAMIENTAS))
grafo.add_edge(START, "modelo")
grafo.add_conditional_edges("modelo", tools_condition)    # va a "tools" o a END
grafo.add_edge("tools", "modelo")                         # el ciclo
agente = grafo.compile()

ver_grafo(agente)
r = agente.invoke({"messages": [("user", "¿Dónde está mi pedido 48213?")]})
mostrar(r["messages"])

---
## Ejercicio 4 — Ver el ciclo nodo por nodo

### Concepto
Con `stream(..., stream_mode="updates")` se observa cada vuelta del ciclo: qué nodo se ejecutó y qué mensaje agregó.

### Ejecutar

In [ ]:
for paso in agente.stream({"messages": [("user", "¿Y el pedido 50555?")]}, stream_mode="updates"):
    for nodo, cambios in paso.items():
        print(f"[{nodo}]")
        mostrar(cambios["messages"])

---
## Ejercicio 5 — Memoria de la conversación

### Concepto
Sin memoria, cada `invoke` empieza de cero. Al compilar con un **checkpointer**, LangGraph guarda los mensajes de cada conversación, identificada por un `thread_id`.

```python
agente_memoria = grafo.compile(checkpointer=InMemorySaver())
hilo = {"configurable": {"thread_id": "ana"}}
```

### Ejecutar

In [ ]:
class EstadoConsulta(TypedDict):
    consulta: str
    categoria: str
    respuesta: str


def clasificar(estado: EstadoConsulta) -> dict:
    palabra = llm.invoke(
        "¿La consulta pregunta por un pedido? Responde solo PEDIDO u OTRO.\n"
        f"Consulta: {estado['consulta']}"
    ).text.strip().upper()
    return {"categoria": palabra}


def responder_pedido(estado: EstadoConsulta) -> dict:
    numero = "".join(ch for ch in estado["consulta"] if ch.isdigit())     # extrae los dígitos
    return {"respuesta": f"Pedido {numero}: {PEDIDOS.get(numero, 'no existe')}."}


def derivar(estado: EstadoConsulta) -> dict:
    return {"respuesta": "Su consulta será atendida por un operador."}


def elegir(estado: EstadoConsulta) -> str:
    return "pedido" if "PEDIDO" in estado["categoria"] else "otro"


grafo = StateGraph(EstadoConsulta)
grafo.add_node("clasificar", clasificar)
grafo.add_node("responder_pedido", responder_pedido)
grafo.add_node("derivar", derivar)
grafo.add_edge(START, "clasificar")
grafo.add_conditional_edges("clasificar", elegir, {"pedido": "responder_pedido", "otro": "derivar"})
grafo.add_edge("responder_pedido", END)
grafo.add_edge("derivar", END)
app_ruta = grafo.compile()

for consulta in ["¿Dónde está mi pedido 77100?", "¿Tienen tienda en Trujillo?"]:
    r = app_ruta.invoke({"consulta": consulta})
    print(f"{consulta}\n  categoría: {r['categoria']} -> {r['respuesta']}\n")

### Práctica
Repetir el turno 2 con otro hilo, `{"configurable": {"thread_id": "luis"}}`, y comprobar que el agente no sabe de qué pedido se habla.